# Yang–Mills on the lattice and the mass gap

*nuclear · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/yang_mills_mass_gap)

The Clay Millennium Problem asks for a rigorous quantum Yang–Mills theory in four dimensions with a mass gap $\Delta > 0$; it is open. Wilson's lattice gauge theory makes the question computable: links $U_\mu(x) \in SU(2)$ (or $U(1)$) stored as unit quaternions are updated by Metropolis for a whole batch of couplings $\beta$ at once. Wilson loops show the area law of confinement, the plaquette matches the exact 2D Bessel ratio and the strong-coupling series, and a glueball correlator gives a lattice mass estimate.

**Vector form**

$$S[U] = \beta\sum_x\sum_{\mu<\nu}\Big(1 - \tfrac12\mathrm{Tr}\,P_{\mu\nu}(x)\Big),\qquad \langle W(R,T)\rangle \sim e^{-\sigma RT},\qquad C(\Delta) = \langle O(t)\,O(t+\Delta)\rangle_c \sim e^{-m\Delta}$$

**Index (tensor) form**

$$P_{\mu\nu}(x) = U_\mu(x)\,U_\nu(x+\hat\mu)\,U_\mu^\dagger(x+\hat\nu)\,U_\nu^\dagger(x),\qquad (ab)_0 = a_0b_0 - a_kb_k,\quad (ab)_k = a_0b_k + b_0a_k + \varepsilon_{kij}a_ib_j,\qquad \Delta S = -\beta\,\tfrac12\mathrm{Tr}\big[(U^\prime_\mu - U_\mu)A_\mu\big]$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $U_\mu(x)$ | `links[:, mu]` | (B, L, …, L, 4) | one unit quaternion per link, for every coupling $\beta_b$ in the batch; $U(1)$ is the subgroup $(\cos\theta, 0, 0, \sin\theta)$ |
| $(ab)_k = a_0b_k + b_0a_k + \varepsilon_{kij}a_ib_j$ | `qmul(a, b)` | (..., 4) | the group law; repeated $i, j$ ⇒ summed (cross product) |
| $U(x+\hat\mu)$ | `torch.roll(U, -1, dims=1 + mu)` | (B, L, …, L, 4) | neighbour on the periodic lattice: every plaquette of every site at once |
| $\tfrac12\mathrm{Tr}\,P_{\mu\nu}$ | `plaquettes(links)` | (B, D(D-1)/2, L, …, L) | the smallest Wilson loop; $\frac12\mathrm{Tr}(ab) = a_0b_0 - a_kb_k$ |
| $A_\mu(x)$ | `staple(links, mu)` | (B, L, …, L, 4) | sum of the $2(D-1)$ staples: the part of $S$ that contains $U_\mu(x)$ is $-\beta\frac12\mathrm{Tr}(U_\mu A_\mu)$ |
| $e^{-\Delta S} > r$ | `ok = (rnd < exp(-dS)) & (par == p)` | (B, L, …, L) | Metropolis on all links of one direction and one checkerboard parity at once (they share no staple) |
| $W(R,T)$ | `wilson_loops(links, 3)` | (B, R, T) | products of link lines around $R\times T$ rectangles, averaged over sites, planes and orientations |
| $u(\beta)$ | `strong_coupling_u(beta, group)` | (B,) | $I_1/I_0$ for $U(1)$, $I_2/I_1$ for $SU(2)$: exact $\langle\frac12\mathrm{Tr}P\rangle$ in 2D, leading order in any $D$ |
| $a\,m_\mathrm{eff}(\Delta)$ | `jackknife(correlator_samples(op, 3), effective_mass)` | (Δ,) | $\ln C(\Delta)/C(\Delta+1)$ of the spatial-plaquette operator, with jackknife errors |

Status: open. The Clay problem (Jaffe and Witten's official description) asks to construct quantum Yang–Mills theory on $\mathbb R^4$ for any compact simple gauge group satisfying axioms such as those of Wightman or Osterwalder–Schrader, and to prove that its spectrum has a gap $\Delta > 0$ above the vacuum. Lattice simulations of $SU(3)$ give a lightest glueball near $1.7$ GeV and confinement (a linear potential, measured string tension) beyond reasonable doubt, but numerical evidence at finite lattice spacing is not a proof. This page works in the same framework at toy size. In 2D the theory is exactly solvable: every plaquette is independent, $\langle\frac12\mathrm{Tr}P\rangle = I_1(\beta)/I_0(\beta)$ for $U(1)$ and $I_2(\beta)/I_1(\beta)$ for $SU(2)$, and $W(R,T) = u^{RT}$ exactly (a test). At strong coupling (small $\beta$) the character expansion gives the same leading values in any dimension and an area law with string tension $\sigma = -\ln u$: confinement is proven there (Osterwalder–Seiler 1978), and also for 3D compact $U(1)$ at every $\beta$ (Göpfert–Mack 1982). The hard part is continuity to weak coupling $\beta \to \infty$ in 4D, where the lattice spacing goes to zero; asymptotic freedom (Gross, Wilczek and Politzer, Nobel Prize 2004) says that is where the continuum limit lies. Creutz ratios $\chi(R,R)$ estimate $\sigma$ from ratios of loops so that perimeter terms cancel. The glueball operator is the sum of spatial plaquettes on a time slice; its connected correlator decays as $e^{-m\Delta}$, and at strong coupling the smallest surface joining two plaquettes is a tube of $4\Delta$ faces, giving $a\,m \approx -4\ln u$. At larger $\beta$ the mass in lattice units falls and needs bigger lattices, smearing and many more sweeps; 4D $U(1)$ has a deconfining transition near $\beta \approx 1.01$ and no mass gap above it, which is why the problem is stated for non-abelian groups. Metropolis with a step $\propto 1/\sqrt{\beta}$ keeps the acceptance near one half; errors use 20 jackknife bins.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/yang_mills_mass_gap/main.py
"""Main loop: parameters → tensors → Monte Carlo → measurements → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import strong_coupling_u, weak_coupling


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    group, dim, size = params["group"], params["dim"], params["size"] // 2 * 2  # even L for the checkerboard
    n_sweeps = 60 if fast_mode() else params["sweeps"]
    n_therm = n_sweeps // 5
    beta0 = params["beta"]
    betas = torch.tensor(tensors.coupling_grid(beta0, group), dtype=torch.float64, device=dev)  # (B,)
    i0 = int((betas - beta0).abs().argmin())
    n_frames = 8 if fast_mode() else 45

    out = solve.simulate(group, dim, size, betas, n_sweeps, n_therm, seed=0, measure_every=10, n_snapshots=n_frames)
    u = strong_coupling_u(betas, group)
    print(
        f"{group} Wilson action on a periodic {size}^{dim} lattice, {len(betas)} couplings in one batch, "
        f"{n_sweeps} Metropolis sweeps ({n_therm} for thermalisation)"
    )
    print(f"acceptance: {out['acceptance'].min():.2f} … {out['acceptance'].max():.2f}")

    # 1. average plaquette against strong and weak coupling
    plaq, plaq_err = solve.jackknife(out["plaquette"], lambda m: m)
    print(f"{'β':>6} {'⟨½TrP⟩':>14} {'strong u(β)':>12} {'weak 1−n/(Dβ)':>14}")
    for b in range(len(betas)):
        print(
            f"{betas[b]:6.3f} {plaq[b]:9.5f}±{plaq_err[b]:.5f} {u[b]:12.5f} {weak_coupling(betas[b], group, dim):14.5f}"
        )

    # 2. Wilson loops and Creutz ratios at the chosen β
    w, w_err = solve.jackknife(out["wilson"][:, i0], lambda m: m)  # (R, T)
    creutz = lambda m: -torch.log(m[1:, 1:] * m[:-1, :-1] / (m[1:, :-1] * m[:-1, 1:]))  # noqa: E731  χ(R, T)
    chi, chi_err = solve.jackknife(out["wilson"][:, i0].clamp_min(1e-12), creutz)
    print(
        f"β = {beta0}: W(R,T) = "
        + ", ".join(f"W({r},{t}) = {w[r - 1, t - 1]:.4f}" for r, t in [(1, 1), (2, 2), (3, 3)])
    )
    print(
        f"Creutz ratio χ(2,2) = {chi[0, 0]:.3f}±{chi_err[0, 0]:.3f} (string tension σa²);"
        f"  strong coupling −ln u = {-math.log(u[i0]):.3f}"
    )

    # 3. glueball: connected correlator of the spatial-plaquette operator between time slices
    t_max = 2  # Δ = 0, 1: beyond that the signal drowns in noise at these statistics
    if out["operator"] is not None:
        samples = solve.correlator_samples(out["operator"][:, i0], t_max)  # (S, Δ + 2)
        mass, mass_err = solve.jackknife(samples, solve.effective_mass)
        lead = -4 * math.log(u[i0])  # strong coupling: the smallest tube between the two plaquettes has 4Δ faces
        print(
            "glueball effective mass a·m_eff(Δ) = ln C(Δ)/C(Δ+1): "
            + ", ".join(f"Δ={d}: {m:.2f}±{e:.2f}" for d, (m, e) in enumerate(zip(mass.tolist(), mass_err.tolist())))
        )
        print(
            f"leading strong-coupling estimate a·m = −4 ln u(β) = {lead:.2f} (lattice estimate, not a continuum proof)"
        )

    running = solve.running(out, i0, n_therm, out["snapshot_sweeps"], t_max, measure_every=10)
    return plot.animate(group, dim, betas, i0, out, plaq_err, u, running, n_therm, out_dir / "yang_mills_mass_gap.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/yang_mills_mass_gap/equation.py
r"""Physics → tensor translation: Wilson's lattice gauge theory with unit-quaternion links.

    S[U] = β Σ_x Σ_{μ<ν} (1 − ½ Tr P_μν(x)),      P_μν(x) = U_μ(x) U_ν(x+μ̂) U_μ(x+ν̂)† U_ν(x)†

Quaternion product (the group law): (ab)₀ = a₀b₀ − aₖbₖ, (ab)ₖ = a₀bₖ + b₀aₖ + ε_kij a_i b_j (k, i, j summed);
½ Tr(ab) = (ab)₀. Shifts x → x + μ̂ are torch.roll on the periodic lattice, so every plaquette, staple and Wilson
loop of the whole batch is a handful of rolls and products. Under a gauge transformation
U_μ(x) → g(x) U_μ(x) g(x+μ̂)† every closed loop of links keeps its trace: the action is gauge invariant.

Strong coupling (character expansion): each plaquette contributes u(β) = I₁(β)/I₀(β) for U(1) and I₂(β)/I₁(β) for
SU(2). In 2D this is exact: ⟨½ Tr P⟩ = u and W(R, T) = u^{RT} (area law, string tension σ = −ln u).
"""

import torch


def qmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Quaternion (group) product of two tensors (..., 4) → (..., 4)."""
    a0, av, b0, bv = a[..., :1], a[..., 1:], b[..., :1], b[..., 1:]
    scalar = a0 * b0 - (av * bv).sum(dim=-1, keepdim=True)  # a₀b₀ − a·b
    cross = torch.linalg.cross(av, bv, dim=-1)  # (a × b)_k = ε_kij a_i b_j
    return torch.cat([scalar, a0 * bv + b0 * av + cross], dim=-1)


def qconj(a: torch.Tensor) -> torch.Tensor:
    """Inverse of a unit quaternion (U†): flip the vector part."""
    return torch.cat([a[..., :1], -a[..., 1:]], dim=-1)


def half_trace(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """½ Tr(ab) = a₀b₀ − a·b without forming the product.   (..., 4) → (...)"""
    return a[..., 0] * b[..., 0] - (a[..., 1:] * b[..., 1:]).sum(dim=-1)


def shift(field: torch.Tensor, mu: int, n: int = 1) -> torch.Tensor:
    """field(x + n μ̂) on the periodic lattice. field (B, L, …, L, 4): site axes are 1 … D."""
    return torch.roll(field, shifts=-n, dims=1 + mu)


def plaquettes(links: torch.Tensor) -> torch.Tensor:
    """½ Tr P_μν(x) for every plane μ < ν and site.   links (B, D, L…, 4) → (B, D(D−1)/2, L…)"""
    dim = links.shape[1]
    out = []
    for mu in range(dim):
        for nu in range(mu + 1, dim):
            u_mu, u_nu = links[:, mu], links[:, nu]
            left = qmul(u_mu, shift(u_nu, mu))  # U_μ(x) U_ν(x+μ̂)
            right = qmul(u_nu, shift(u_mu, nu))  # U_ν(x) U_μ(x+ν̂)
            out.append(half_trace(left, qconj(right)))  # ½ Tr[U_μ U_ν(x+μ̂) (U_ν U_μ(x+ν̂))†]
    return torch.stack(out, dim=1)


def staple(links: torch.Tensor, mu: int) -> torch.Tensor:
    """A_μ(x) = Σ_{ν≠μ} [U_ν(x+μ̂) U_μ(x+ν̂)† U_ν(x)† + U_ν(x+μ̂−ν̂)† U_μ(x−ν̂)† U_ν(x−ν̂)], so the part of S that
    contains U_μ(x) is −β ½ Tr(U_μ(x) A_μ(x)).   (B, D, L…, 4) → (B, L…, 4)"""
    u_mu = links[:, mu]
    total = torch.zeros_like(u_mu)
    for nu in range(links.shape[1]):
        if nu == mu:
            continue
        u_nu = links[:, nu]
        up = qmul(qmul(shift(u_nu, mu), qconj(shift(u_mu, nu))), qconj(u_nu))
        down_nu = shift(u_nu, nu, -1)  # U_ν(x−ν̂)
        down = qmul(qmul(qconj(shift(down_nu, mu)), qconj(shift(u_mu, nu, -1))), down_nu)
        total = total + up + down
    return total


def gauge_transform(links: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
    """U_μ(x) → g(x) U_μ(x) g(x+μ̂)† for every μ.   links (B, D, L…, 4), g (B, L…, 4)"""
    return torch.stack([qmul(qmul(g, links[:, mu]), qconj(shift(g, mu))) for mu in range(links.shape[1])], dim=1)


def line(links: torch.Tensor, mu: int, length: int) -> torch.Tensor:
    """Ordered product U_μ(x) U_μ(x+μ̂) ⋯ U_μ(x+(n−1)μ̂) starting at every site.   → (B, L…, 4)"""
    out = links[:, mu]
    for step in range(1, length):
        out = qmul(out, shift(links[:, mu], mu, step))
    return out


def wilson_loops(links: torch.Tensor, r_max: int) -> torch.Tensor:
    """W(R, T) = ⟨½ Tr⟩ of the R × T rectangle, averaged over sites, planes and both orientations.   → (B, R, T)"""
    dim = links.shape[1]
    lines = {(mu, n): line(links, mu, n) for mu in range(dim) for n in range(1, r_max + 1)}
    w = torch.zeros(links.shape[0], r_max, r_max, dtype=links.dtype, device=links.device)
    for mu in range(dim):
        for nu in range(dim):
            if mu == nu:
                continue
            for r in range(1, r_max + 1):
                for t in range(1, r_max + 1):
                    lower = qmul(lines[mu, r], shift(lines[nu, t], mu, r))  # along μ, then up ν
                    upper = qmul(lines[nu, t], shift(lines[mu, r], nu, t))  # up ν, then along μ
                    w[:, r - 1, t - 1] += half_trace(lower, qconj(upper)).flatten(1).mean(dim=1)
    return w / (dim * (dim - 1))


def strong_coupling_u(beta: torch.Tensor, group: str) -> torch.Tensor:
    """Character coefficient u(β): I₁/I₀ for U(1), I₂/I₁ = I₀/I₁ − 2/β for SU(2) (scaled Bessels, no overflow)."""
    i0, i1 = torch.special.i0e(beta), torch.special.i1e(beta)
    return i1 / i0 if group == "U(1)" else i0 / i1 - 2 / beta


def weak_coupling(beta: torch.Tensor, group: str, dim: int) -> torch.Tensor:
    """Equipartition at large β: each of the (N²−1)(D−1) physical link modes per site carries ½, so
    ⟨½ Tr P⟩ ≈ 1 − n_gen/(D β) with n_gen = 3 (SU(2)) or 1 (U(1))."""
    n_gen = 3 if group == "SU(2)" else 1
    return 1 - n_gen / (dim * beta)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/yang_mills_mass_gap/tensors.py
"""Build the concrete tensors: random group elements as unit quaternions, hot-start lattices and the checkerboard.

A link variable U_μ(x) ∈ SU(2) is stored as a unit quaternion (a₀, a₁, a₂, a₃), U = a₀ + i aₖσₖ with ½ Tr U = a₀.
U(1) is the subgroup (cos θ, 0, 0, sin θ): products stay in it, and ½ Tr U = cos θ gives the U(1) Wilson action,
so the same code runs both groups. A lattice is a tensor (B, D, L, …, L, 4): batch of couplings β_b, link direction μ,
D site coordinates, quaternion component.
"""

import math

import torch


def random_su2(shape: tuple, gen: torch.Generator, device: torch.device) -> torch.Tensor:
    """Haar-random SU(2): a normalised 4D Gaussian is uniform on the 3-sphere S³ ≅ SU(2).   → (*shape, 4)"""
    q = torch.randn(*shape, 4, generator=gen, dtype=torch.float64).to(device)
    return q / q.norm(dim=-1, keepdim=True)


def random_u1(shape: tuple, gen: torch.Generator, device: torch.device) -> torch.Tensor:
    """Haar-random U(1) phase e^{iθ} as the quaternion (cos θ, 0, 0, sin θ).   → (*shape, 4)"""
    theta = (2 * math.pi * torch.rand(*shape, generator=gen, dtype=torch.float64)).to(device)
    zero = torch.zeros_like(theta)
    return torch.stack([torch.cos(theta), zero, zero, torch.sin(theta)], dim=-1)


def random_element(shape: tuple, group: str, gen: torch.Generator, device: torch.device) -> torch.Tensor:
    return random_su2(shape, gen, device) if group == "SU(2)" else random_u1(shape, gen, device)


def hot_start(n_beta: int, dim: int, size: int, group: str, gen: torch.Generator, device: torch.device) -> torch.Tensor:
    """Every link random (infinite temperature, β = 0): (B, D, L, …, L, 4)."""
    return random_element((n_beta, dim) + (size,) * dim, group, gen, device)


def random_gauge(shape: tuple, group: str, gen: torch.Generator, device: torch.device) -> torch.Tensor:
    """A gauge transformation g(x), one group element per site: (B, L, …, L, 4)."""
    return random_element(shape, group, gen, device)


def parity(dim: int, size: int, device: torch.device) -> torch.Tensor:
    """(x₁ + … + x_D) mod 2 for every site: (L, …, L). Links of one direction on one parity share no staple."""
    coords = torch.meshgrid(*[torch.arange(size, device=device)] * dim, indexing="ij")
    return sum(coords) % 2


def coupling_grid(beta: float, group: str) -> list[float]:
    """β values simulated side by side (one batch); the chosen β is added so it gets the long analysis."""
    top = 4.0 if group == "SU(2)" else 2.0
    grid = [round(top * k / 6, 3) for k in range(1, 7)]
    return sorted(set(grid + [beta]))


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/yang_mills_mass_gap/solve.py
"""Metropolis Monte Carlo for the whole batch of couplings at once (checkerboard over links), measurements, and the
glueball correlator with jackknife errors."""

import torch

from problems._common.progress import steps

from . import tensors
from .equation import half_trace, plaquettes, qmul, staple, wilson_loops


def proposal(shape: tuple, eps: torch.Tensor, group: str, gen: torch.Generator) -> torch.Tensor:
    """Random group element near 1, R = normalise(1, ε·n) with Gaussian n; R and R⁻¹ are equally likely, so the
    Metropolis step obeys detailed balance. eps (B,) one step size per coupling.   → (*shape, 4)"""
    n = torch.randn(*shape, 4, generator=gen, dtype=torch.float64).to(eps.device)
    if group == "U(1)":
        n[..., 1:3] = 0.0  # stay on the U(1) circle (cos θ, 0, 0, sin θ)
    n[..., 1:] *= eps.view(-1, *([1] * len(shape)))  # shape = (B, L, …): step ε_b for coupling b
    n[..., 0] = 1.0
    return n / n.norm(dim=-1, keepdim=True)


def sweep(links: torch.Tensor, beta: torch.Tensor, group: str, gen: torch.Generator, hits: int = 2):
    """One sweep: for every direction μ and parity, all those links get `hits` Metropolis trials at once.
    Returns the new links and the acceptance rate per coupling (B,)."""
    links = links.clone()
    dim, size = links.shape[1], links.shape[2]
    par = tensors.parity(dim, size, links.device)
    eps = (1.5 / torch.sqrt(beta * (dim - 1))).clamp(max=10.0)  # step ∝ 1/√(β·staple size): acceptance ≈ ½
    bshape = (-1,) + (1,) * dim
    accepted = torch.zeros_like(beta)
    for mu in range(dim):
        for p in (0, 1):
            a = staple(links, mu)  # (B, L…, 4) fixed while the μ-links of parity p change
            old = links[:, mu]
            s_old = half_trace(old, a)
            for _ in range(hits):
                new = qmul(proposal(old.shape[:-1], eps, group, gen), old)
                new = new / new.norm(dim=-1, keepdim=True)
                s_new = half_trace(new, a)
                d_action = -beta.view(bshape) * (s_new - s_old)  # ΔS = −β ½Tr[(U′ − U) A]
                rnd = torch.rand(s_new.shape, generator=gen, dtype=torch.float64).to(links.device)
                ok = (rnd < torch.exp(-d_action)) & (par == p)
                old = torch.where(ok[..., None], new, old)
                s_old = torch.where(ok, s_new, s_old)
                accepted += ok.flatten(1).sum(dim=1) / ok[0].numel()
            links[:, mu] = old
    return links, accepted / (dim * hits)


def simulate(
    group: str,
    dim: int,
    size: int,
    betas: torch.Tensor,
    n_sweeps: int,
    n_therm: int,
    seed: int,
    measure_every: int,
    r_max: int = 3,
    n_snapshots: int = 0,
) -> dict:
    """Hot start, n_therm sweeps of thermalisation, then measure every sweep:
    plaquette (S, B); glueball operator O(t) = Σ_x spatial ½ Tr P on each time slice (S, B, L), axis 0 is time;
    Wilson loops every measure_every sweeps (M, B, R, T); n_snapshots plaquette slices (K, B, L, L)."""
    gen = torch.Generator().manual_seed(seed)
    links = tensors.hot_start(len(betas), dim, size, group, gen, betas.device)
    plaq, ops, loops, snaps, snap_sweep, acc = [], [], [], [], [], []
    snap_at = {round(k * (n_sweeps - 1) / max(1, n_snapshots - 1)) for k in range(n_snapshots)}
    spatial = [i for i, (mu, nu) in enumerate((m, n) for m in range(dim) for n in range(m + 1, dim)) if mu > 0]
    for s in steps(n_sweeps, "sweeps"):
        links, a = sweep(links, betas, group, gen)
        acc.append(a)
        p = plaquettes(links)  # (B, P, L…)
        if s in snap_at:
            snaps.append(p[:, 0].reshape(len(betas), size, size, -1)[..., 0])  # plane (0, 1) through x₂ = 0
            snap_sweep.append(s)
        if s < n_therm:
            continue
        plaq.append(p.flatten(1).mean(dim=1))
        if dim > 2:
            ops.append(p[:, spatial].sum(dim=1).flatten(2).sum(dim=2))  # (B, L): sum over spatial sites
        if (s - n_therm) % measure_every == 0:
            loops.append(wilson_loops(links, r_max))
    return {
        "plaquette": torch.stack(plaq),
        "operator": torch.stack(ops) if ops else None,
        "wilson": torch.stack(loops),
        "snapshots": torch.stack(snaps) if snaps else None,
        "snapshot_sweeps": snap_sweep,
        "acceptance": torch.stack(acc).mean(dim=0),
    }


def jackknife(samples: torch.Tensor, estimator, n_bins: int = 20) -> tuple[torch.Tensor, torch.Tensor]:
    """Mean and error of estimator(bin means) by leave-one-bin-out jackknife (bins absorb autocorrelation).
    samples (S, …); estimator maps a (…) tensor of means to the quantity."""
    n = samples.shape[0] // n_bins * n_bins
    bins = samples[:n].reshape(n_bins, n // n_bins, *samples.shape[1:]).mean(dim=1)  # (n_bins, …)
    total = bins.sum(dim=0)
    leave_out = torch.stack([estimator((total - bins[i]) / (n_bins - 1)) for i in range(n_bins)])
    centre = estimator(bins.mean(dim=0))
    err = torch.sqrt((n_bins - 1) * ((leave_out - leave_out.mean(dim=0)) ** 2).mean(dim=0))
    return centre, err


def correlator_samples(op: torch.Tensor, t_max: int) -> torch.Tensor:
    """Per-configuration ingredients of C(Δ) = ⟨O(t)O(t+Δ)⟩ − ⟨O⟩²: [Ō, mean_t O(t)O(t+Δ) for Δ = 0 … t_max].
    op (S, B, L) → (S, B, t_max + 2)"""
    prods = [(op * torch.roll(op, -d, dims=-1)).mean(dim=-1) for d in range(t_max + 1)]
    return torch.stack([op.mean(dim=-1)] + prods, dim=-1)


def effective_mass(means: torch.Tensor) -> torch.Tensor:
    """m_eff(Δ) = ln C(Δ)/C(Δ+1) from the averaged ingredients (…, t_max + 2) → (…, t_max)."""
    c = means[..., 1:] - means[..., :1] ** 2  # connected correlator C(Δ)
    return torch.log(c[..., :-1].clamp_min(1e-30) / c[..., 1:].clamp_min(1e-30))


def running(out: dict, i0: int, n_therm: int, frames: list[int], t_max: int, measure_every: int = 5) -> dict:
    """The estimates as they would have looked after each frame's sweep: ⟨½TrP⟩ (K, B), W(R, T) at β_{i0}
    (K, R, T) and the glueball m_eff(Δ) with jackknife errors (K, Δ); NaN before enough data exist."""
    plaq, wil, mass, err = [], [], [], []
    nan = float("nan")
    for s in frames:
        n = s - n_therm + 1  # measurements so far (one per sweep after thermalisation)
        n_w = (s - n_therm) // measure_every + 1
        plaq.append(out["plaquette"][:n].mean(dim=0) if n > 0 else torch.full_like(out["plaquette"][0], nan))
        wil.append(out["wilson"][:n_w, i0].mean(dim=0) if n > 0 else torch.full_like(out["wilson"][0, i0], nan))
        if out["operator"] is not None and n >= 40:
            m, e = jackknife(correlator_samples(out["operator"][:n, i0], t_max), effective_mass)
        else:
            m = e = torch.full((t_max,), nan, dtype=torch.float64)
        mass.append(m)
        err.append(e)
    return {
        "plaquette": torch.stack(plaq),
        "wilson": torch.stack(wil),
        "mass": torch.stack(mass),
        "mass_err": torch.stack(err),
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/yang_mills_mass_gap/plot.py
"""A slice of the plaquette field, ⟨plaquette⟩(β) against strong/weak coupling, the Wilson-loop area law and the
glueball effective mass, as the Monte Carlo statistics accumulate."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import strong_coupling_u, weak_coupling


def animate(group, dim, betas, i0, out, plaq_err, u, running, n_therm, path: Path) -> Path:
    b = betas.cpu().numpy()
    snaps = out["snapshots"][:, i0].cpu().numpy()  # (K, L, L)
    sweeps = out["snapshot_sweeps"]
    rp, rw = running["plaquette"].cpu().numpy(), running["wilson"].cpu().numpy()
    rm, re = running["mass"].cpu().numpy(), running["mass_err"].cpu().numpy()
    perr = plaq_err.cpu().numpy()
    u0 = float(u[i0])

    fig, (ax_f, ax_p, ax_w, ax_m) = plt.subplots(
        1, 4, figsize=(12.5, 3.5), gridspec_kw={"width_ratios": [1, 1.2, 1.1, 1.1]}
    )
    img = ax_f.imshow(snaps[0].T, origin="lower", cmap="RdBu_r", vmin=-1, vmax=1)
    ax_f.grid(False)
    ax_f.set_xlabel("$x_0$ (time)")
    ax_f.set_ylabel("$x_1$")
    fig.colorbar(img, ax=ax_f, fraction=0.046, pad=0.03)
    f_title = ax_f.set_title("", fontsize=9)

    fine = torch.linspace(0.05, float(b.max()) * 1.05, 200, dtype=torch.float64)
    ax_p.plot(
        fine, strong_coupling_u(fine, group), color="#9ca3af", ls="--", lw=1.2, label="strong coupling $u(\\beta)$"
    )
    weak = fine[weak_coupling(fine, group, dim) > 0.35]
    ax_p.plot(weak, weak_coupling(weak, group, dim), color="#9ca3af", ls=":", lw=1.4, label=r"weak $1 - n/(D\beta)$")
    pts = ax_p.errorbar(b, rp[-1], yerr=perr, fmt="o", ms=4, color=ACCENT[0], label="Monte Carlo")
    ax_p.set_ylim(0, 1)
    ax_p.set_xlabel(r"$\beta$")
    ax_p.set_title(r"$\langle\frac{1}{2}\mathrm{Tr}\,P\rangle$" + f", {group}, D = {dim}", fontsize=9)
    ax_p.legend(fontsize=7, loc="lower right")

    r_max = rw.shape[-1]
    area = np.array([[r * t for t in range(1, r_max + 1)] for r in range(1, r_max + 1)], dtype=float)
    a_line = np.linspace(0, area.max() + 0.5, 50)
    ax_w.plot(a_line, -np.log(u0) * a_line, color="#9ca3af", ls="--", lw=1.2, label=r"area law $-RT\ln u$")
    scat = ax_w.scatter(area.ravel(), np.zeros(area.size), s=22, color=ACCENT[1], zorder=3, label="$-\\ln W(R,T)$")
    ax_w.set_xlabel("area $RT$")
    ax_w.legend(fontsize=7, loc="upper left")
    ax_w.set_title(rf"Wilson loops, $\beta = {b[i0]:g}$", fontsize=9)
    ax_w.set_ylim(0, -np.log(u0) * area.max() * 1.15)

    n_delta = rm.shape[-1]
    lead = -4 * np.log(u0)
    ax_m.axhline(lead, color="#9ca3af", ls="--", lw=1.2, label=r"strong coupling $-4\ln u$")
    m_bars = [ax_m.errorbar([], [], yerr=[], fmt="s", color=ACCENT[2], ms=5, label=r"$a\,m_\mathrm{eff}(\Delta)$")]
    ax_m.set_xlim(-0.5, n_delta - 0.5)
    ax_m.set_xticks(range(n_delta))
    ax_m.set_ylim(0, lead * 1.8 + 0.5)
    ax_m.set_xlabel(r"$\Delta$ (time slices)")
    ax_m.legend(fontsize=7, loc="upper right")
    if out["operator"] is None:
        ax_m.text(
            0.5,
            0.5,
            "no glueball operator in 2D:\nno propagating gluons",
            ha="center",
            transform=ax_m.transAxes,
            fontsize=8,
        )
    m_title = ax_m.set_title("", fontsize=9)

    def update(f: int):
        img.set_data(snaps[f].T)
        phase = "thermalising" if sweeps[f] < n_therm else "measuring"
        f_title.set_text(rf"$\frac{{1}}{{2}}$Tr$P_{{01}}$, sweep {sweeps[f] + 1} ({phase})")
        y = rp[f]
        for line in pts.lines[2]:
            line.set_alpha(1.0 if f == len(sweeps) - 1 else 0.0)  # error bars once all data are in
        pts.lines[0].set_data(b, y)
        w = rw[f]
        loops = np.where(w.ravel() > 3e-3, w.ravel(), np.nan)  # loops below ~3e-3 are lost in the noise
        scat.set_offsets(np.c_[area.ravel(), -np.log(loops)])
        m_bars[0].remove()
        m_bars[0] = ax_m.errorbar(np.arange(n_delta), rm[f], yerr=re[f], fmt="s", color=ACCENT[2], ms=5, capsize=3)
        ok = np.isfinite(rm[f, 0])
        m_title.set_text(
            rf"glueball: $a\,m_\mathrm{{eff}}(0) = {rm[f, 0]:.2f} \pm {re[f, 0]:.2f}$"
            if ok
            else "glueball correlator (collecting)"
        )

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(sweeps), len(sweeps)), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'group': 'SU(2)',
    'dim': 3,
    'size': 8,
    'beta': 2.0,
    'sweeps': 600
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.yang_mills_mass_gap")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.yang_mills_mass_gap.main")
problem_dir = Path("problems/nuclear/yang_mills_mass_gap")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
